# AI-Powered Robo Advisor — Model Training Notebook
**Course Project — Build AI with AWS | Week 4**

This notebook:
1. Pulls historical OHLCV data for the top 50 companies
2. Engineers simple features and a trading-signal target
3. Trains and compares two ML models
4. Plots cumulative strategy return vs. actual (buy-and-hold) return
5. Saves the best model so it can be uploaded to S3 and used by our Lambda function


In [ ]:
# If running for the first time, uncomment to install dependencies
# !pip install yfinance scikit-learn pandas matplotlib joblib --quiet


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import os

os.makedirs("models", exist_ok=True)
os.makedirs("plots", exist_ok=True)


## 1. Define the universe (Top 50 companies)

In [ ]:
# Top 50 large-cap US tickers (edit freely to match your chosen list)
TICKERS = [
    "AAPL","MSFT","GOOGL","AMZN","NVDA","META","TSLA","BRK-B","LLY","AVGO",
    "JPM","V","UNH","XOM","MA","PG","HD","COST","MRK","ABBV",
    "PEP","KO","ADBE","WMT","CRM","BAC","NFLX","AMD","TMO","MCD",
    "CSCO","ACN","LIN","ABT","DHR","ORCL","INTC","TXN","PM","NKE",
    "WFC","DIS","VZ","CMCSA","INTU","IBM","AMGN","CAT","GE","QCOM"
]
print(f"Universe size: {len(TICKERS)}")


## 2. Pull historical OHLCV data

In [ ]:
data = yf.download(TICKERS, period="3y", interval="1d", group_by="ticker", auto_adjust=True, progress=False)
data.to_csv("raw_ohlcv_top50.csv")
print(data.shape)
data.head()


**Note:** save `raw_ohlcv_top50.csv` and later upload it to your S3 `raw-data/` folder — this satisfies the "AWS Infrastructure Setup" storage requirement.

## 3. Feature engineering & target (using one ticker to illustrate — repeat/loop for all 50)

In [ ]:
def build_features(df):
    df = df.copy()
    df["return_1d"] = df["Close"].pct_change()
    df["sma_10"] = df["Close"].rolling(10).mean()
    df["sma_30"] = df["Close"].rolling(30).mean()
    df["volatility_10"] = df["return_1d"].rolling(10).std()
    df["momentum_10"] = df["Close"] - df["Close"].shift(10)
    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)  # 1 = price goes up next day
    return df.dropna()

ticker = "AAPL"
df = data[ticker].copy()
feat_df = build_features(df)
feat_df.tail()


## 4. Train / test split

In [ ]:
FEATURES = ["return_1d", "sma_10", "sma_30", "volatility_10", "momentum_10"]
X = feat_df[FEATURES]
y = feat_df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print(X_train.shape, X_test.shape)


## 5. Train and compare models

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=500),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = {"model": model, "accuracy": acc, "preds": preds}
    print(f"{name}: accuracy = {acc:.4f}")


## 6. Backtest: cumulative strategy return vs. actual (buy-and-hold) return

In [ ]:
best_name = max(results, key=lambda n: results[n]["accuracy"])
best_preds = results[best_name]["preds"]
print(f"Best model: {best_name}")

test_df = feat_df.iloc[-len(y_test):].copy()
test_df["signal"] = best_preds  # 1 = long, 0 = flat
test_df["strategy_return"] = test_df["return_1d"].shift(-1) * test_df["signal"]
test_df["actual_return"] = test_df["return_1d"].shift(-1)

test_df["cum_strategy"] = (1 + test_df["strategy_return"].fillna(0)).cumprod()
test_df["cum_actual"] = (1 + test_df["actual_return"].fillna(0)).cumprod()

plt.figure(figsize=(10, 5))
plt.plot(test_df.index, test_df["cum_actual"], label="Actual (Buy & Hold)")
plt.plot(test_df.index, test_df["cum_strategy"], label=f"Strategy ({best_name})")
plt.title(f"Cumulative Return — {ticker}")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.tight_layout()
plt.savefig("plots/cumulative_return_comparison.png", dpi=150)
plt.show()


This PNG (`plots/cumulative_return_comparison.png`) is one of your required deliverables — a cumulative return plot comparing actual vs. strategy returns.

## 7. Save the best model (to upload to S3 for Lambda)

In [ ]:
joblib.dump(results[best_name]["model"], "models/trading_model.joblib")
print("Saved models/trading_model.joblib — this is your full model artifact for the README / notebook deliverable")


## 7b. Export a lightweight version for Lambda (Logistic Regression coefficients only)

AWS Lambda has a 250MB size limit for a function plus its layers, and scikit-learn's dependencies
(numpy + scipy) can push you right up against that limit. To keep the deployment simple and reliable,
we export just the **Logistic Regression** model's coefficients as a small JSON file. Lambda then computes
predictions with plain Python — no scikit-learn, joblib, or numpy needed inside Lambda at all.

This is a common real-world pattern: keep the full model (with all candidates you compared) in your notebook
and README for evaluation, but deploy the simplest model that meets your accuracy bar.

In [ ]:
import json

lr_model = results["LogisticRegression"]["model"]

model_params = {
    "feature_order": FEATURES,
    "coefficients": lr_model.coef_[0].tolist(),
    "intercept": float(lr_model.intercept_[0]),
}

with open("models/model_params.json", "w") as f:
    json.dump(model_params, f, indent=2)

print("Saved models/model_params.json — upload this (not the .joblib) to your S3 models/ folder")
print(json.dumps(model_params, indent=2))


## 8. Next steps
1. Upload `raw_ohlcv_top50.csv` to S3 `raw-data/`
2. Upload `models/model_params.json` to S3 `models/` (this is what Lambda will load — small and dependency-free)
3. Keep `models/trading_model.joblib` for your own reference / README, but Lambda does not need it
4. Move on to building the Lambda function that loads `model_params.json` and returns a recommendation
